# 03 — Mesh: full-field DIC

A `Mesh` covers an entire region of interest with a triangulated network of nodes.
Each node is a `Subset` centre — the solver correlates all of them using a
**reliability-guided** strategy: start from a "seed" node with a known good initial
guess, then propagate outward in order of correlation quality.

This notebook covers: `Mesh` construction, geometry, `solve()`, displacement
contours, ZNCC score maps, and save/load.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import geopyv_dev as gp

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "animation.embed_limit": 50,
})


In [ ]:
# Generate a reference and a deformed speckle image.
# 5 px x-translation + slight shear — fast enough to run in < 2 s.
os.makedirs("data/single", exist_ok=True)
speckle = gp.Speckle(
    image_cfg={"image_dir": "data/single", "name": "demo", "image_size": (401, 401)},
    speckle_cfg={"speckle_size": 3.5, "speckle_number": 700},
    progression="deformation",
    deformation_cfg={"comp": [5.0, 0.0, 0.005, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
    noise_cfg=(0.0, 1.5),
    scale_cfg={"scale": "lin", "n": 2},
)
speckle.solve(seed=42)
f_img = gp.Image(filepath="data/single/demo_0.jpg")
g_img = gp.Image(filepath="data/single/demo_1.jpg")
print(f"Reference : {f_img}")
print(f"Target    : {g_img}")

boundary_nodes = np.array([
    [50.0, 50.0], [50.0, 350.0], [350.0, 350.0], [350.0, 50.0]
])
boundary  = gp.PathRegion(nodes=boundary_nodes, hard=False)
local_mask = gp.Mask(mask_type="local", shape="circle", size=25)
seed_coord = [200.0, 200.0]


## Constructing a `Mesh`

```python
mesh = gp.Mesh(
    boundary    = boundary,       # PathRegion, CircleRegion, or (N,2) array
    target_nodes = 80,            # approximate number of nodes (solver binary-searches for element size)
    f_img       = f_img,          # reference Image
    g_img       = g_img,          # target Image
    size        = (10., 80.),     # (min, max) element edge length in pixels
    exclusions  = [excl_region],  # optional holes in the mesh
    mesh_order  = 2,              # 1 = linear triangles; 2 = quadratic (mid-side nodes)
)
```

The mesh geometry is computed immediately at construction using a constrained
Delaunay triangulation (spade).  No correlation happens yet.

### Linear vs quadratic elements

| `mesh_order` | Nodes per element | DOF per node | Best for |
|---|---|---|---|
| 1 | 3 (vertices) | 6 | Large deformations, coarse grids |
| 2 | 6 (vertices + mid-sides) | 12 | Strain gradients, fine grids |


In [ ]:
# Build two meshes: linear (order 1) and quadratic (order 2)
mesh1 = gp.Mesh(boundary=boundary, target_nodes=60, f_img=f_img, g_img=g_img,
                 size=(15., 70.), mesh_order=1)
mesh2 = gp.Mesh(boundary=boundary, target_nodes=60, f_img=f_img, g_img=g_img,
                 size=(15., 70.), mesh_order=2)

print(f"Order-1  : {mesh1}")
print(f"Order-2  : {mesh2}")
print(f"\nOrder-1 has {mesh1.nodes.shape[0]} nodes, {mesh1.elements.shape[0]} elements")
print(f"Order-2 has {mesh2.nodes.shape[0]} nodes, {mesh2.elements.shape[0]} elements")


In [ ]:
# Visualise the mesh geometry before solving
def plot_mesh_geometry(mesh, ax, title, img):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255, alpha=0.55)
    nodes = mesh.nodes
    elems = mesh.elements
    # Draw element edges
    for elem in elems:
        corners = elem[:3]  # first 3 nodes are always vertices
        tri_pts = nodes[corners]
        tri_closed = np.vstack([tri_pts, tri_pts[0]])
        ax.plot(tri_closed[:, 0], tri_closed[:, 1], "b-", lw=0.6, alpha=0.6)
    # Draw nodes
    ax.scatter(nodes[:, 0], nodes[:, 1], s=12, c="crimson", zorder=5, linewidths=0)
    # Mark boundary nodes
    bn = nodes[mesh.boundary]
    ax.scatter(bn[:, 0], bn[:, 1], s=30, c="navy", marker="s", zorder=6,
               linewidths=0, label="boundary")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.axis("off")
    ax.legend(fontsize=8, loc="lower right")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_mesh_geometry(mesh1, ax1, f"Linear mesh  (order 1)  –  {mesh1.nodes.shape[0]} nodes",
                   f_img.image_gs)
plot_mesh_geometry(mesh2, ax2, f"Quadratic mesh (order 2) –  {mesh2.nodes.shape[0]} nodes",
                   f_img.image_gs)
plt.tight_layout()
os.makedirs("../docs/assets", exist_ok=True)
fig.savefig("../docs/assets/nb3_mesh_geometry.png", dpi=120, bbox_inches="tight")
plt.show()


## Reliability-guided propagation

The solver does **not** correlate all nodes independently.  Instead it:

1. Solves the **seed node** (the node closest to `seed_coord`) with a tight tolerance.
2. Ranks all unsolved neighbours by their C_ZNCC (best first).
3. Uses the best neighbour's warp as an initial guess for the next node.
4. Repeats until all nodes are solved or fail the `tolerance` threshold.

This dramatically improves robustness — each node gets a warm start from an already-converged
neighbour rather than starting from zero.

The animation below illustrates the concept.


In [ ]:
# Animated propagation concept
# We approximate the solve order by BFS distance from the seed node.
import collections

nodes_arr = mesh1.nodes
elems_arr = mesh1.elements

# Build adjacency list (vertex-to-vertex)
n_nodes = nodes_arr.shape[0]
adj = collections.defaultdict(set)
for elem in elems_arr:
    verts = list(elem[:3])
    for i, a in enumerate(verts):
        for b in verts[i+1:]:
            adj[a].add(b); adj[b].add(a)

# BFS from seed node
seed_node = int(np.argmin(np.linalg.norm(nodes_arr - np.array([200., 200.]), axis=1)))
order, visited, queue = [], {seed_node}, collections.deque([seed_node])
while queue:
    n = queue.popleft()
    order.append(n)
    for nb_n in sorted(adj[n]):
        if nb_n not in visited:
            visited.add(nb_n); queue.append(nb_n)

# Animate nodes appearing in BFS order
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(f_img.image_gs, cmap="gray", vmin=0, vmax=255, alpha=0.5)
for elem in elems_arr:
    tri = nodes_arr[elem[:3]]
    ax.plot(*np.append(tri, [tri[0]], axis=0).T, "b-", lw=0.5, alpha=0.35)

scatter_solved  = ax.scatter([], [], s=35, c="seagreen",  zorder=5, linewidths=0, label="solved")
scatter_pending = ax.scatter(nodes_arr[:, 0], nodes_arr[:, 1],
                              s=35, c="#bbb", zorder=4, linewidths=0, label="pending")
seed_mark = ax.scatter([nodes_arr[seed_node, 0]], [nodes_arr[seed_node, 1]],
                        s=100, c="crimson", zorder=6, marker="*", label="seed")

frame_label = ax.text(0.02, 0.97, "", transform=ax.transAxes,
                       va="top", fontsize=10, color="black",
                       bbox=dict(fc="white", ec="none", alpha=0.7))
ax.legend(fontsize=9, loc="lower right")
ax.axis("off")
ax.set_title("Reliability-guided propagation (BFS order)", fontsize=11, fontweight="bold")

STEP = max(1, len(order) // 40)   # ~40 animation frames

def update_prop(frame):
    n_done = min((frame + 1) * STEP, len(order))
    done = order[:n_done]
    pending = order[n_done:]
    scatter_solved.set_offsets(nodes_arr[done])
    scatter_pending.set_offsets(nodes_arr[pending] if pending else np.empty((0, 2)))
    frame_label.set_text(f"{n_done}/{len(order)} nodes solved")
    return scatter_solved, scatter_pending, frame_label

anim_prop = FuncAnimation(fig, update_prop, frames=40, interval=120, blit=True)
plt.tight_layout()
anim_prop.save("../docs/assets/nb3_propagation.gif", writer="pillow", fps=10)
HTML(anim_prop.to_jshtml())


In [ ]:
# Solve the mesh  (takes a few seconds)
mesh1.solve(
    local_mask    = local_mask,
    seed_coord    = seed_coord,
    seed_warp     = [0.0] * 6,
    subset_order  = 1,
    tolerance     = 0.75,
    seed_tolerance= 0.9,
    method        = "icgn",
)
print(mesh1)
print(f"\n  mean C_ZNCC : {mesh1.c_zncc.mean():.4f}")
print(f"  min  C_ZNCC : {mesh1.c_zncc.min():.4f}")
print(f"  seed node   : {mesh1.seed_node}")


In [ ]:
# Displacement field — u (horizontal) and v (vertical)
nodes = mesh1.nodes
disps = mesh1.displacements

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
titles = ["u-displacement (px)", "v-displacement (px)", "C_ZNCC score"]
data   = [disps[:, 0], disps[:, 1], mesh1.c_zncc]
cmaps  = ["RdBu_r", "RdBu_r", "RdYlGn"]

for ax, d, title, cmap in zip(axes, data, titles, cmaps):
    ax.imshow(f_img.image_gs, cmap="gray", vmin=0, vmax=255, alpha=0.4)
    sc = ax.scatter(nodes[:, 0], nodes[:, 1], c=d, cmap=cmap, s=30,
                    linewidths=0, vmin=d.min(), vmax=d.max())
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=11, fontweight="bold"); ax.axis("off")

plt.suptitle("Mesh solution — colour by per-node value", fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig("../docs/assets/nb3_displacement.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Save and reload the mesh
mesh1.save("data/mesh.pyv")
mesh_loaded = gp.load("data/mesh.pyv")
print(f"Loaded: {mesh_loaded}")
print(f"  mean C_ZNCC : {mesh_loaded.c_zncc.mean():.4f}  (matches original)")


## Key takeaways

- `target_nodes` controls mesh density; `size` bounds element edge lengths.
- `mesh_order=2` (quadratic) uses more nodes but captures strain gradients better.
- Reliability-guided propagation makes the solver robust to initial-guess errors.
- Always check `mesh.c_zncc.min()` — values below ~0.7 indicate problem nodes.

**Next:** Notebook 04 — extending to a full image sequence with `Sequence`.
